# Assignment 4 End-to-end: raw file to model-ready data

**Dataset source:** https://www.kaggle.com/datasets/amitabhajoy/bengaluru-house-price-data  *(Bengaluru House Price Data, file `Bengaluru_House_Data.csv`)*

Goal: take the raw listings file to a clean `X_train, X_test, y_train, y_test`. No model is trained here.
Target: `price` (in lakhs of rupees).

In [1]:
import re
import numpy as np
import pandas as pd
import joblib
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

RAW_PATH = "Bengaluru_House_Data.csv"
DATASET_LINK = "https://www.kaggle.com/datasets/amitabhajoy/bengaluru-house-price-data"
RANDOM_STATE = 42

## The four functions
`load_data()`, `clean(df)`, `add_features(df)` and `build_pipeline()` each sit in their own cell (Q4).
`load_data()` comes first so that the data dictionary and the quality report can call it. The rest of the notebook only calls these functions.

In [2]:
def load_data():
    """Read the raw CSV exactly as it is, with no changes."""
    return pd.read_csv(RAW_PATH)

## Q1. Data dictionary

In [3]:
raw = load_data()
print("raw shape:", raw.shape)

data_dictionary = pd.DataFrame([
    ["area_type",    "How the area of the property is measured / type of property", "category (text)", "n/a",           "Super built-up Area, Built-up Area, Carpet Area, Plot Area"],
    ["availability", "'Ready To Move' or the month the property will be available",  "text (mixed)",    "n/a",           "'Ready To Move' or a month such as '19-Dec'"],
    ["location",     "Neighbourhood in Bengaluru",                                  "category (text)", "n/a",           "about 1,300 different neighbourhoods"],
    ["size",         "Number of bedrooms, written as '2 BHK' or '4 Bedroom'",        "text",            "bedrooms",      "1 to 10 bedrooms"],
    ["society",      "Name of the housing society / project",                       "text",            "n/a",           "any name, often missing"],
    ["total_sqft",   "Total area of the property",                                  "text (mixed)",    "square feet",   "about 300 to 10,000 sq ft (a few larger plots)"],
    ["bath",         "Number of bathrooms",                                         "number",          "count",         "1 to 10"],
    ["balcony",      "Number of balconies",                                         "number",          "count",         "0 to 3"],
    ["price",        "Listing price (TARGET)",                                      "number",          "lakhs of rupees (1 lakh = 100,000)", "about 10 to 2,000"],
], columns=["column", "meaning", "type", "unit", "expected range"])
data_dictionary

raw shape: (13320, 9)


,column,meaning,type,unit,expected range
0,area_type,How the area of the property is measured / typ...,category (text),n/a,"Super built-up Area, Built-up Area, Carpet Are..."
1,availability,'Ready To Move' or the month the property will...,text (mixed),n/a,'Ready To Move' or a month such as '19-Dec'
2,location,Neighbourhood in Bengaluru,category (text),n/a,"about 1,300 different neighbourhoods"
3,size,"Number of bedrooms, written as '2 BHK' or '4 B...",text,bedrooms,1 to 10 bedrooms
4,society,Name of the housing society / project,text,n/a,"any name, often missing"
5,total_sqft,Total area of the property,text (mixed),square feet,"about 300 to 10,000 sq ft (a few larger plots)"
6,bath,Number of bathrooms,number,count,1 to 10
7,balcony,Number of balconies,number,count,0 to 3
8,price,Listing price (TARGET),number,"lakhs of rupees (1 lakh = 100,000)","about 10 to 2,000"


## Q2. Raw quality report (with a count for each problem)

In [4]:
sqft_txt = raw["total_sqft"].astype(str)
is_range = sqft_txt.str.fullmatch(r"\s*[\d.]+\s*-\s*[\d.]+\s*")
has_unit = sqft_txt.str.contains(r"[A-Za-z]")
plain_sqft = pd.to_numeric(raw["total_sqft"], errors="coerce")

bhk_raw = pd.to_numeric(raw["size"].str.extract(r"(\d+)", expand=False), errors="coerce")
area_clean = raw["area_type"].str.split().str.join(" ")
size_txt = raw["size"].fillna("")
avail_is_date = ~raw["availability"].str.strip().str.lower().eq("ready to move")

problems = [
    ("missing values", "society",  raw["society"].isna().sum()),
    ("missing values", "balcony",  raw["balcony"].isna().sum()),
    ("missing values", "bath",     raw["bath"].isna().sum()),
    ("missing values", "size",     raw["size"].isna().sum()),
    ("missing values", "location", raw["location"].isna().sum()),
    ("duplicates", "exact duplicate rows", raw.duplicated().sum()),
    ("inconsistent units", "total_sqft written as a range ('1200 - 2400')", is_range.sum()),
    ("inconsistent units", "total_sqft in other units (Sq. Meter, Perch, Yards ...)", (has_unit & ~is_range).sum()),
    ("inconsistent labels", "area_type with stray extra spaces", (raw["area_type"] != area_clean).sum()),
    ("inconsistent labels", "size written as 'Bedroom' (others use 'BHK')", size_txt.str.contains("Bedroom").sum()),
    ("inconsistent labels", "size written as 'RK'", size_txt.str.contains("RK").sum()),
    ("inconsistent labels", "availability holds a date instead of 'Ready To Move'", avail_is_date.sum()),
    ("impossible / implausible", "10 or more bathrooms", (raw["bath"] >= 10).sum()),
    ("impossible / implausible", "10 or more bedrooms", (bhk_raw >= 10).sum()),
    ("impossible / implausible", "bathrooms > bedrooms + 2", (raw["bath"] > bhk_raw + 2).sum()),
    ("impossible / implausible", "under 300 sq ft per bedroom (plain-number rows only)", ((plain_sqft / bhk_raw) < 300).sum()),
]
quality_report = pd.DataFrame(problems, columns=["problem type", "detail", "count"])
quality_report

,problem type,detail,count
0,missing values,society,5502
1,missing values,balcony,609
2,missing values,bath,73
3,missing values,size,16
4,missing values,location,1
5,duplicates,exact duplicate rows,529
6,inconsistent units,total_sqft written as a range ('1200 - 2400'),201
7,inconsistent units,"total_sqft in other units (Sq. Meter, Perch, Y...",46
8,inconsistent labels,area_type with stray extra spaces,13320
9,inconsistent labels,size written as 'Bedroom' (others use 'BHK'),2525


## Q3. New features (and one rejected idea)

| feature | built from | one-line reasoning |
|---|---|---|
| `bhk` | `size` | The text "2 BHK" / "4 Bedroom" cannot be used by a model; the bedroom count is the number that matters. |
| `sqft_per_bhk` | `sqft`, `bhk` | Space per bedroom separates a roomy flat from a cramped one better than total area alone. |
| `bath_per_bhk` | `bath`, `bhk` | Bathrooms relative to bedrooms flags luxury homes and data-entry errors (14 baths for 4 bedrooms). |
| `is_ready_to_move` | `availability` | The column mixes the label "Ready To Move" with dates; a simple yes/no keeps the useful part. |
| `has_society` | `society` | `society` is mostly missing with unusable names, but whether a listing belongs to a named society may still carry information. |

**Rejected:** `price_per_sqft`. It is built from the target (`price`), so using it as an input would leak the answer into the features.

### `clean(df)`

In [5]:
LOG = {}      # numbers behind every cleaning decision (used in Q9)

SQFT_UNITS = {                 # square feet in one unit
    "sq. meter": 10.7639, "sq. yards": 9.0, "perch": 272.25,
    "acres": 43560.0, "cents": 435.6, "guntha": 1089.0, "grounds": 2400.0,
}

def to_sqft(value):
    """'1200' -> 1200 | '1200 - 2400' -> 1800 | '34.46Sq. Meter' -> 370.9 | anything else -> NaN"""
    s = str(value).strip().lower()
    m = re.fullmatch(r"([\d.]+)\s*-\s*([\d.]+)", s)
    if m:
        return (float(m.group(1)) + float(m.group(2))) / 2
    m = re.fullmatch(r"([\d.]+)\s*([a-z. ]+)", s)
    if m and m.group(2).strip() in SQFT_UNITS:
        return float(m.group(1)) * SQFT_UNITS[m.group(2).strip()]
    try:
        return float(s)
    except ValueError:
        return np.nan


def clean(df):
    df = df.copy()
    LOG.clear()
    LOG["raw_rows"] = len(df)

    # 1. exact duplicates
    LOG["duplicates_dropped"] = int(df.duplicated().sum())
    df = df.drop_duplicates()

    # 2. text labels
    df["area_type"] = df["area_type"].str.split().str.join(" ")
    df["location"] = df["location"].str.strip()

    # 3. total_sqft -> one numeric column in square feet
    sqft_txt = df["total_sqft"].astype(str)
    LOG["sqft_ranges_averaged"] = int(sqft_txt.str.fullmatch(r"\s*[\d.]+\s*-\s*[\d.]+\s*").sum())
    LOG["sqft_unit_converted"] = int((sqft_txt.str.contains(r"[A-Za-z]")).sum())
    df["sqft"] = df["total_sqft"].map(to_sqft)

    # 4. size -> number of bedrooms
    df["bhk"] = pd.to_numeric(df["size"].str.extract(r"(\d+)", expand=False), errors="coerce")
    df = df.drop(columns=["total_sqft", "size"])

    # 5. rows that cannot be repaired
    LOG["society_missing"] = int(df["society"].isna().sum())
    LOG["society_missing_pct"] = round(100 * df["society"].isna().mean(), 1)
    core = df[["location", "bhk", "sqft"]].isna().any(axis=1)
    LOG["dropped_missing_core"] = int(core.sum())
    df = df[~core]

    # 6. impossible values become missing; implausible rows are dropped (domain rules)
    df.loc[df["bath"] <= 0, "bath"] = np.nan
    df = df[df["price"] > 0]
    df = df[df["sqft"] > 0]
    too_small = (df["sqft"] / df["bhk"]) < 300           # under 300 sq ft per bedroom
    LOG["dropped_sqft_per_bhk"] = int(too_small.sum())
    df = df[~too_small]
    too_many_baths = df["bath"] > df["bhk"] + 2          # e.g. 14 bathrooms for 4 bedrooms
    LOG["dropped_bath_gt_bhk_plus2"] = int(too_many_baths.sum())
    df = df[~too_many_baths]

    # 7. what is left to impute later (inside the pipeline)
    LOG["balcony_missing"] = int(df["balcony"].isna().sum())
    LOG["balcony_missing_pct"] = round(100 * df["balcony"].isna().mean(), 1)
    LOG["bath_missing"] = int(df["bath"].isna().sum())
    LOG["bath_missing_pct"] = round(100 * df["bath"].isna().mean(), 1)
    LOG["clean_rows"] = len(df)
    return df.reset_index(drop=True)

### `add_features(df)`

In [6]:
def add_features(df):
    df = df.copy()
    df["sqft_per_bhk"] = df["sqft"] / df["bhk"]
    df["bath_per_bhk"] = df["bath"] / df["bhk"]
    df["is_ready_to_move"] = (df["availability"].str.strip().str.lower() == "ready to move").astype(int)
    df["has_society"] = df["society"].notna().astype(int)
    return df.drop(columns=["availability", "society"])

### `build_pipeline()`

In [7]:
NUMERIC = ["sqft", "bhk", "bath", "balcony", "sqft_per_bhk", "bath_per_bhk"]
BINARY = ["is_ready_to_move", "has_society"]

def build_pipeline():
    numeric_steps = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    preprocess = ColumnTransformer([
        ("num", numeric_steps, NUMERIC),
        ("bin", "passthrough", BINARY),
        ("area", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["area_type"]),
        # locations seen fewer than 10 times in train share one 'infrequent' column;
        # a location never seen in train is also sent there
        ("loc", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=10,
                              sparse_output=False), ["location"]),
    ])
    return Pipeline([("preprocess", preprocess)])

## Q5. The whole path, raw file to `X_train, X_test, y_train, y_test`

In [8]:
raw = load_data()
df_clean = clean(raw)
df_feat = add_features(df_clean)

X = df_feat.drop(columns="price")
y = df_feat["price"]
X_train_df, X_test_df, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

pipe = build_pipeline()
X_train = pipe.fit_transform(X_train_df)      # fitted on train only
X_test = pipe.transform(X_test_df)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (9621, 209)
X_test : (2406, 209)
y_train: (9621,)
y_test : (2406,)


## Q6. Proof that the output is clean

In [9]:
assert not np.isnan(X_train).any(), "NaN left in X_train"
assert not np.isnan(X_test).any(), "NaN left in X_test"
assert not y_train.isna().any() and not y_test.isna().any(), "NaN left in the target"
assert X_train.shape[1] == X_test.shape[1], "train and test have different column counts"
assert (X_train.std(axis=0) > 0).all(), "a constant column exists"
print("OK: no NaN, same number of columns in train and test (%d), no constant column" % X_train.shape[1])

OK: no NaN, same number of columns in train and test (209), no constant column


## Q7. Before and after

In [10]:
final_matrix = np.vstack([X_train, X_test])

before_after = pd.DataFrame({
    "Raw file": [raw.shape[0], raw.shape[1], int(raw.isna().sum().sum()), int(raw.duplicated().sum())],
    "Final": [final_matrix.shape[0], final_matrix.shape[1], int(np.isnan(final_matrix).sum()),
              int(df_feat.duplicated().sum())],
}, index=["row count", "column count", "missing value count", "duplicate count"])
before_after

,Raw file,Final
row count,13320,12027
column count,9,209
missing value count,6201,0
duplicate count,529,97


*Final* duplicate count is measured on the cleaned data (including `price`). Final column count is the width of the model-ready matrix after one-hot encoding.

## Q8. Save the cleaned CSV and the fitted pipeline

In [11]:
df_feat.to_csv("bengaluru_clean.csv", index=False)
joblib.dump(pipe, "pipeline.joblib")
print("saved: bengaluru_clean.csv and pipeline.joblib")

try:
    from google.colab import files
    files.download("bengaluru_clean.csv")
    files.download("pipeline.joblib")
except ImportError:
    print("Not running in Colab; use the file panel to download both files")

saved: bengaluru_clean.csv and pipeline.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Q9. Every decision and the number behind it


- Dropped 529 exact duplicate rows (4.0% of 13320 raw rows)
- Dropped the society column, 41.7% missing (5328 rows); kept only a has_society yes/no flag
- Averaged 200 total_sqft ranges into one number (e.g. '1200 - 2400' -> 1800)
- Converted 46 total_sqft values given in other units (Sq. Meter, Perch, ...) to square feet
- Dropped 17 rows missing location, size or a readable total_sqft (cannot be repaired)
- Dropped 738 rows with under 300 sq ft per bedroom (implausible space)
- Dropped 9 rows with more than bedrooms + 2 bathrooms (implausible)
- Filled balcony (496 missing, 4.1%) and bath (57 missing, 0.5%) with the train median, inside the pipeline
- Dropped availability after turning it into is_ready_to_move (the column mixes 'Ready To Move' and dates)
- One-hot encoded 1118 train locations into 197 columns; locations seen fewer than 10 times share one 'infrequent' column
- Did not create price_per_sqft because it is built from the target (leakage)
- Split 9621 train / 2406 test rows (80/20, random_state=42) before fitting any imputer, scaler or encoder

In [12]:
loc_enc = pipe.named_steps["preprocess"].named_transformers_["loc"]
n_locations = len(loc_enc.categories_[0])
n_loc_cols = len(loc_enc.get_feature_names_out())
L = LOG
decisions = [
    f"Dropped {L['duplicates_dropped']} exact duplicate rows ({100 * L['duplicates_dropped'] / L['raw_rows']:.1f}% of {L['raw_rows']} raw rows)",
    f"Dropped the society column, {L['society_missing_pct']}% missing ({L['society_missing']} rows); kept only a has_society yes/no flag",
    f"Averaged {L['sqft_ranges_averaged']} total_sqft ranges into one number (e.g. '1200 - 2400' -> 1800)",
    f"Converted {L['sqft_unit_converted']} total_sqft values given in other units (Sq. Meter, Perch, ...) to square feet",
    f"Dropped {L['dropped_missing_core']} rows missing location, size or a readable total_sqft (cannot be repaired)",
    f"Dropped {L['dropped_sqft_per_bhk']} rows with under 300 sq ft per bedroom (implausible space)",
    f"Dropped {L['dropped_bath_gt_bhk_plus2']} rows with more than bedrooms + 2 bathrooms (implausible)",
    f"Filled balcony ({L['balcony_missing']} missing, {L['balcony_missing_pct']}%) and bath ({L['bath_missing']} missing, {L['bath_missing_pct']}%) with the train median, inside the pipeline",
    f"Dropped availability after turning it into is_ready_to_move (the column mixes 'Ready To Move' and dates)",
    f"One-hot encoded {n_locations} train locations into {n_loc_cols} columns; locations seen fewer than 10 times share one 'infrequent' column",
    f"Did not create price_per_sqft because it is built from the target (leakage)",
    f"Split {len(X_train)} train / {len(X_test)} test rows (80/20, random_state={RANDOM_STATE}) before fitting any imputer, scaler or encoder",
]
print("\n".join("- " + d for d in decisions))

- Dropped 529 exact duplicate rows (4.0% of 13320 raw rows)
- Dropped the society column, 41.7% missing (5328 rows); kept only a has_society yes/no flag
- Averaged 200 total_sqft ranges into one number (e.g. '1200 - 2400' -> 1800)
- Converted 46 total_sqft values given in other units (Sq. Meter, Perch, ...) to square feet
- Dropped 17 rows missing location, size or a readable total_sqft (cannot be repaired)
- Dropped 738 rows with under 300 sq ft per bedroom (implausible space)
- Dropped 9 rows with more than bedrooms + 2 bathrooms (implausible)
- Filled balcony (496 missing, 4.1%) and bath (57 missing, 0.5%) with the train median, inside the pipeline
- Dropped availability after turning it into is_ready_to_move (the column mixes 'Ready To Move' and dates)
- One-hot encoded 1118 train locations into 197 columns; locations seen fewer than 10 times share one 'infrequent' column
- Did not create price_per_sqft because it is built from the target (leakage)
- Split 9621 train / 2406 test ro

## Q10. What is still wrong with this dataset

Several problems could not be fixed. `area_type` mixes super built-up, built-up, carpet and plot area, so the square-feet figure does not mean the same thing in every row, and nothing in the file says how to convert between them. `society` is missing in 41.7% of rows and the names that exist look scrambled, so I dropped it and kept only a yes/no flag. There is no listing date, so prices from different years cannot be compared or adjusted. I averaged the square-feet ranges and used a 300 sq ft per bedroom rule that removed 738 rows; these are my assumptions, so some real homes may have been dropped while some bad rows remain. Finally, about 920 of the 1,118 locations in the training data appear fewer than 10 times, so the model has very little evidence about most neighbourhoods and treats them as one shared group.

## README.md

In [13]:
ba = before_after
readme = f'''# ML Assignment 4 End-to-end pipeline

Notebook: `assignment4_endtoend.ipynb` (run top to bottom with Runtime -> Restart and run all).
Files: `{RAW_PATH}` (raw data), `bengaluru_clean.csv` (cleaned data), `pipeline.joblib` (fitted pipeline).

## Dataset
Bengaluru House Price Data. Target: `price` (lakhs of rupees). No model is trained.
Source: {DATASET_LINK}

## Before and after
| | Raw file | Final |
|---|---|---|
| row count | {ba.loc['row count', 'Raw file']} | {ba.loc['row count', 'Final']} |
| column count | {ba.loc['column count', 'Raw file']} | {ba.loc['column count', 'Final']} |
| missing value count | {ba.loc['missing value count', 'Raw file']} | {ba.loc['missing value count', 'Final']} |
| duplicate count | {ba.loc['duplicate count', 'Raw file']} | {ba.loc['duplicate count', 'Final']} |

Final duplicate count is measured on the cleaned data; final column count is the width of the model-ready matrix.

Shapes: X_train {X_train.shape}, X_test {X_test.shape}, y_train {y_train.shape}, y_test {y_test.shape}
'''
with open("README.md", "w") as f:
    f.write(readme)
print(readme)

# ML Assignment 4 End-to-end pipeline

Notebook: `assignment4_endtoend.ipynb` (run top to bottom with Runtime -> Restart and run all).
Files: `Bengaluru_House_Data.csv` (raw data), `bengaluru_clean.csv` (cleaned data), `pipeline.joblib` (fitted pipeline).

## Dataset
Bengaluru House Price Data. Target: `price` (lakhs of rupees). No model is trained.
Source: https://www.kaggle.com/datasets/amitabhajoy/bengaluru-house-price-data

## Before and after
| | Raw file | Final |
|---|---|---|
| row count | 13320 | 12027 |
| column count | 9 | 209 |
| missing value count | 6201 | 0 |
| duplicate count | 529 | 97 |

Final duplicate count is measured on the cleaned data; final column count is the width of the model-ready matrix.

Shapes: X_train (9621, 209), X_test (2406, 209), y_train (9621,), y_test (2406,)

